# Grounded Image & Video Captioning — One-Click Notebook

Run the single cell below to execute the complete pipeline in **one click**.

### What this cell does:
1. **Auto-Setup**: Detects Google Colab, clones the repository, and auto-downloads the trained relation model checkpoint (`relation_mlp.pt`).
2. **Auto-Detect Input**: Automatically scans the **`input/`** folder for images (`.jpg`, `.png`, etc.) or videos (`.mp4`, `.mov`, etc.).
3. **Proposed Model Execution**:
   - **If Image**: Runs YOLO detection + CLIP verification + Visual-Semantic Relation prediction + Grounded Reranking. Outputs side-by-side **Baseline vs Proposed Model** captions and displays the annotated image inline.
   - **If Video**: Samples keyframes, grounds interactions per second, outputs timestamped transcripts, and embeds a playable annotated video directly in the notebook.

> **💡 How to use your own input (Drag & Drop)**:
> In the left sidebar of Google Colab, click **`📁 Files`** → open **`input/`** → drag and drop your image or video, then click Run!

In [ ]:
# ==============================================================================
# ONE-CLICK GROUNDED CAPTIONING PIPELINE (IMAGE & VIDEO)
# ==============================================================================
import os
import sys
import json
import time
import base64
import urllib.request
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import torch
from IPython.display import display, HTML

# ------------------------------------------------------------------------------
# 1. Environment Setup (Google Colab & Local)
# ------------------------------------------------------------------------------
if "google.colab" in sys.modules:
    colab_dir = "/content/Grounded-Image-Captioning"
    if not os.path.exists(colab_dir):
        print("🌐 Cloning repository into Google Colab...")
        os.system("git clone -b caption-rerank https://github.com/SmaranReddy/Grounded-Image-Captioning.git /content/Grounded-Image-Captioning")
    os.chdir(colab_dir)
    if colab_dir not in sys.path:
        sys.path.insert(0, colab_dir)
    print("Installing dependencies...")
    os.system("pip install -q ultralytics transformers accelerate")
else:
    proj_root = str(Path(".").resolve())
    if proj_root not in sys.path:
        sys.path.insert(0, proj_root)

# Ensure trained relation checkpoint weights exist (auto-download if missing in Colab)
rel_ckpt = Path("checkpoints_gpu/geometry_clip_union_seed43/relation_mlp.pt")
if not rel_ckpt.exists() or rel_ckpt.stat().st_size < 1000:
    print("📥 Downloading trained relation checkpoint weights (2.6 MB)...")
    rel_ckpt.parent.mkdir(parents=True, exist_ok=True)
    ckpt_url = "https://raw.githubusercontent.com/SmaranReddy/Grounded-Image-Captioning/caption-rerank/checkpoints_gpu/geometry_clip_union_seed43/relation_mlp.pt"
    urllib.request.urlretrieve(ckpt_url, str(rel_ckpt))
    print("✓ Relation checkpoint verified and ready!")

# Ensure results_caption configs exist
for cfg_name in ["rerank_weights.json", "relation_checkpoint_selection.json"]:
    cfg_file = Path("results_caption") / cfg_name
    if not cfg_file.exists():
        cfg_file.parent.mkdir(parents=True, exist_ok=True)
        cfg_url = f"https://raw.githubusercontent.com/SmaranReddy/Grounded-Image-Captioning/caption-rerank/results_caption/{cfg_name}"
        urllib.request.urlretrieve(cfg_url, str(cfg_file))

INPUT_DIR = Path("input")
OUTPUT_DIR = Path("results")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Device: {device.upper()}")
if device == "cuda":
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("  (Tip: In Colab, go to Runtime > Change runtime type > T4 GPU for faster processing)")

# Import pipeline modules
import demo_single_image as demo
from src.inference.video_pipeline import run_video_pipeline
from src.utils.visualize import draw_relation_boxes

# ------------------------------------------------------------------------------
# 2. Input Selection (Set path or leave empty "" to auto-detect latest input)
# ------------------------------------------------------------------------------
INPUT_FILE = ""  # Leave empty "" to auto-pick latest file in 'input/', or set e.g. "input/image.jpg" or "input/sample1.mp4"
SAMPLE_RATE = 1.0  # (For videos) Frame sampling interval in seconds

img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
vid_exts = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

if INPUT_FILE:
    target_path = Path(INPUT_FILE)
else:
    all_media = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in img_exts.union(vid_exts)]
    if not all_media:
        raise FileNotFoundError(
            "No image or video found in 'input/' folder. "
            "Please drag & drop a file into 'input/' via Colab's left sidebar and run again!"
        )
    all_media.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    target_path = all_media[0]

suffix = target_path.suffix.lower()
print(f"\n📁 Processing Target: {target_path} (Format: {suffix})")

# ------------------------------------------------------------------------------
# 3. Execute Pipeline According to Media Type
# ------------------------------------------------------------------------------
if suffix in img_exts:
    # ========================== IMAGE WORKFLOW ==========================
    class DemoArgs:
        image = str(target_path)
        device = device
        selection = "results_caption/relation_checkpoint_selection.json"
        weights = "results_caption/rerank_weights.json"
        blip_dtype = "float32"
        num_candidates = 4
        conf = 0.5
        out_dir = str(OUTPUT_DIR)

    demo_res = demo.run(DemoArgs())
    demo_out_path = demo.write_result(demo_res, str(OUTPUT_DIR))

    baseline_cap = demo_res.get("captions", {}).get("baseline", "N/A")
    proposed_cap = (
        demo_res.get("captions", {}).get("relation_reranked")
        or demo_res.get("captions", {}).get("object_reranked")
        or baseline_cap
    )
    sel_rel = demo_res.get("relation", {}).get("selected")
    rel_str = f"{sel_rel['subject']} {sel_rel['predicate']} {sel_rel['object']} ({sel_rel.get('confidence', 0):.2f})" if sel_rel else "None"

    print("\n" + "=" * 75)
    print("  GROUNDED IMAGE CAPTIONING: BASELINE vs PROPOSED MODEL")
    print("=" * 75)
    print(f"  🔴 Baseline Caption (Vanilla BLIP) : {baseline_cap}")
    print(f"  🟢 Proposed Caption (Our Model)    : {proposed_cap}")
    print(f"  🔗 Predicted Relation Grounding    : {rel_str}")
    print("=" * 75)
    print("\nDetailed Verification & Reranking Report:")
    print(demo.render_report(demo_res))

    # Draw visual annotations
    vis_path = OUTPUT_DIR / f"{target_path.stem}_annotated.jpg"
    img_pil = Image.open(target_path).convert("RGB")
    draw_relation_boxes(
        img_pil,
        demo_res.get("detections", {}).get("verified", []),
        [sel_rel] if sel_rel else [],
        str(vis_path),
        caption=proposed_cap,
    )

    # Display inline
    plt.figure(figsize=(10, 8))
    plt.imshow(Image.open(vis_path))
    plt.axis("off")
    plt.title(f"PROPOSED MODEL CAPTION:\n{proposed_cap}\n(Baseline: {baseline_cap})", fontsize=11, pad=12)
    plt.tight_layout()
    plt.show()

elif suffix in vid_exts:
    # ========================== VIDEO WORKFLOW ==========================
    transcript = run_video_pipeline(
        video_path=target_path,
        output_dir=OUTPUT_DIR,
        sample_rate=SAMPLE_RATE,
        render_video=True,
        top_k_relations=3,
        temperature=2.0,
        compare_baseline=True,
    )

    print("\n" + "=" * 75)
    print(f"  GROUNDED VIDEO CAPTIONING TRANSCRIPT ({len(transcript)} keyframe(s))")
    print("=" * 75)
    for frame in transcript:
        t_str = frame.get("timestamp_str", f"{frame.get('timestamp_sec', 0):.1f}s")
        rels = ", ".join(frame.get("relations", [])) or "none"
        print(f"[{t_str}]")
        print(f"  🔗 Relations        : {rels}")
        print(f"  🔴 Baseline Caption : {frame.get('baseline_caption')}")
        print(f"  🟢 Proposed Caption : {frame.get('caption')}\n")

    annotated_video_path = OUTPUT_DIR / f"{target_path.stem}_annotated.mp4"
    if annotated_video_path.exists():
        print(f"✓ Annotated video saved to: {annotated_video_path}")
        try:
            with open(annotated_video_path, "rb") as f:
                vid_b64 = base64.b64encode(f.read()).decode("ascii")
            display(HTML(f'''
                <video width="640" controls autoplay muted>
                    <source src="data:video/mp4;base64,{vid_b64}" type="video/mp4">
                    Your browser does not support HTML5 video.
                </video>
            '''))
        except Exception as e:
            print(f"(Video playback preview skipped: {e})")
else:
    raise ValueError(f"Unsupported file extension: {suffix}")
